# Notebook 04 — Fine-tuning LLaMA 3.1 8B avec Unsloth + LoRA

**Objectif** : Fine-tuner `unsloth/Meta-Llama-3.1-8B-bnb-4bit` sur le corpus Q&R avec LoRA (rank=16), puis lancer l'inférence sur le test set.

> ⚠️ **GPU OBLIGATOIRE** — Aller dans `Exécution > Modifier le type d'exécution > GPU (T4 ou A100)`

**Modèle** : LLaMA 3.1 8B — même modèle que Baseline et RAG (03) → comparaison équitable des 4 méthodes.

**Inputs** :
- `BASE_PATH/data/processed/train.json`
- `BASE_PATH/data/processed/test.json`

**Outputs** :
- `BASE_PATH/models/lora_adapter/` — adaptateur LoRA
- `BASE_PATH/results/finetuned_predictions.json`

## 0. Vérification GPU

In [ ]:
# Vérification que le GPU est bien disponible avant de continuer
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "Aucun GPU détecté !\n"
        "→ Allez dans Exécution > Modifier le type d'exécution > GPU, puis relancez."
    )

gpu_name   = torch.cuda.get_device_name(0)
gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU détecté  : {gpu_name}")
print(f"VRAM totale  : {gpu_mem_gb:.1f} Go")
print(f"CUDA version : {torch.version.cuda}")

## 1. Montage Google Drive

In [ ]:
# Montage du Drive et définition du chemin de base du projet
from google.colab import drive
drive.mount('/content/drive')

BASE_PATH = '/content/drive/MyDrive/llm-integration-study/'

## 2. Installation des dépendances

In [ ]:
# Installation d'Unsloth (version optimisée pour Colab) — peut prendre 2-3 minutes
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

In [ ]:
# Installation des bibliothèques de fine-tuning
!pip install -q trl peft transformers accelerate bitsandbytes

## 3. Imports et configuration

In [ ]:
# Imports de toutes les bibliothèques nécessaires
import os
import json
import time
import numpy as np
import torch
from tqdm.notebook import tqdm
from datasets import Dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments

# Chemins Drive
PROCESSED_PATH = os.path.join(BASE_PATH, 'data', 'processed')
MODELS_PATH    = os.path.join(BASE_PATH, 'models', 'lora_adapter')
RESULTS_PATH   = os.path.join(BASE_PATH, 'results')

for path in [MODELS_PATH, RESULTS_PATH]:
    os.makedirs(path, exist_ok=True)

# LLaMA 3.1 8B — même modèle que Baseline/RAG → comparaison équitable
BASE_MODEL   = "unsloth/Meta-Llama-3.1-8B-bnb-4bit"
LORA_RANK    = 16
LORA_ALPHA   = 16
LORA_DROPOUT = 0
# Modules clés de LLaMA 3.1 (attention + MLP) pour un fine-tuning efficace
TARGET_MODS  = ["q_proj", "k_proj", "v_proj", "o_proj",
                "gate_proj", "up_proj", "down_proj"]
MAX_SEQ_LEN  = 1024   # réduit de 2048 → économise ~30% VRAM

# Hyperparamètres d'entraînement
NUM_EPOCHS   = 3
BATCH_SIZE   = 2      # réduit de 4 → 2 pour éviter OOM sur GPU 15 Go
LR           = 2e-4

# Libère la VRAM fragmentée avant l'entraînement
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

print("Configuration :")
print(f"  Modèle de base   : {BASE_MODEL}")
print(f"  LoRA rank/alpha  : {LORA_RANK}/{LORA_ALPHA}")
print(f"  Modules LoRA     : {TARGET_MODS}")
print(f"  Époques          : {NUM_EPOCHS}")
print(f"  Batch size       : {BATCH_SIZE}")
print(f"  Learning rate    : {LR}")

## 4. Chargement des données

In [ ]:
# Chargement de train.json et test.json depuis Drive
def load_json(path):
    """Charge un fichier JSON avec gestion d'erreur."""
    try:
        with open(path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        print(f"  Chargé : {path} ({len(data)} entrées)")
        return data
    except FileNotFoundError:
        print(f"  [ERROR] Fichier introuvable : {path}")
        print("  → Assurez-vous d'avoir exécuté 02_dataset_builder.ipynb au préalable.")
        return []
    except json.JSONDecodeError as e:
        print(f"  [ERROR] JSON invalide : {e}")
        return []

print("Chargement des datasets...")
train_data = load_json(os.path.join(PROCESSED_PATH, 'train.json'))
test_data  = load_json(os.path.join(PROCESSED_PATH, 'test.json'))

print(f"\nTrain : {len(train_data)} paires")
print(f"Test  : {len(test_data)} questions")

## 5. Formatage Alpaca

In [ ]:
# Formatage du dataset au format Alpaca pour le SFTTrainer
ALPACA_TEMPLATE = (
    "### Instruction: Réponds à cette question.\n"
    "### Input: {question}\n"
    "### Response: {answer}"
)

def format_alpaca(item):
    """Convertit une paire Q&R au format Alpaca."""
    return {
        "text": ALPACA_TEMPLATE.format(
            question=item.get('question', '').strip(),
            answer=item.get('answer', '').strip()
        )
    }

# Conversion vers HuggingFace Dataset
formatted_train = [format_alpaca(item) for item in train_data if item.get('question') and item.get('answer')]
hf_dataset = Dataset.from_list(formatted_train)

print(f"Dataset Alpaca prêt : {len(hf_dataset)} exemples")
print("\nAperçu du premier exemple :")
print(hf_dataset[0]['text'][:300])

## 6. Chargement du modèle et application de LoRA

In [ ]:
# Chargement de Mistral-7B quantifié en 4-bit via Unsloth
print(f"Chargement de {BASE_MODEL}...")
print("(Téléchargement ~4 Go depuis HuggingFace, peut prendre 5-10 min)")

try:
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL,
        max_seq_length=MAX_SEQ_LEN,
        dtype=None,          # Auto-détection (bfloat16 sur A100, float16 sur T4)
        load_in_4bit=True,   # Quantification 4-bit pour tenir en VRAM
    )
    print("Modèle de base chargé.")
except Exception as e:
    raise RuntimeError(f"Échec du chargement du modèle : {e}")

In [ ]:
# Application de l'adaptateur LoRA sur les couches cibles
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=TARGET_MODS,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing="unsloth",  # Économie mémoire supplémentaire
    random_state=42,
    use_rslora=False,
    loftq_config=None,
)

# Comptage des paramètres entraînables
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Paramètres entraînables : {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

## 7. Entraînement

In [ ]:
# Configuration du SFTTrainer avec les hyperparamètres définis
training_args = TrainingArguments(
    output_dir="/content/tmp_checkpoints",
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=8,       # Effective batch = 2×8 = 16 (identique)
    warmup_steps=10,
    learning_rate=LR,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    optim="adamw_8bit",                  # Optimiseur 8-bit pour économiser la VRAM
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=42,
    report_to="none",                    # Pas de Wandb
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=hf_dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    dataset_num_proc=2,
    args=training_args,
)

print("Trainer configuré. Démarrage de l'entraînement...")

In [ ]:
# Lancement de l'entraînement (environ 20-40 min sur T4 pour 300 exemples × 3 époques)
train_start = time.time()

try:
    trainer_stats = trainer.train()
    train_duration = time.time() - train_start
    print(f"\nEntraînement terminé en {train_duration/60:.1f} minutes")
    print(f"Loss finale      : {trainer_stats.training_loss:.4f}")
    print(f"Steps total      : {trainer_stats.global_step}")
except Exception as e:
    raise RuntimeError(f"Échec de l'entraînement : {e}")

## 8. Sauvegarde de l'adaptateur LoRA sur Drive

In [ ]:
# Sauvegarde de l'adaptateur LoRA (uniquement les poids delta, ~50 Mo)
try:
    model.save_pretrained(MODELS_PATH)
    tokenizer.save_pretrained(MODELS_PATH)
    print(f"Adaptateur LoRA sauvegardé : {MODELS_PATH}")

    # Listage des fichiers produits
    files = os.listdir(MODELS_PATH)
    for fname in files:
        fpath = os.path.join(MODELS_PATH, fname)
        size  = os.path.getsize(fpath) / 1024
        print(f"  {fname:<40} {size:>8.1f} Ko")
except Exception as e:
    print(f"[ERROR] Sauvegarde LoRA : {e}")

## 9. Inférence sur le test set

In [ ]:
# Passage en mode inférence (désactive le gradient pour accélérer)
FastLanguageModel.for_inference(model)

# Tokens Alpaca qui signalent la fin de la réponse
_ALPACA_STOP_MARKERS = ["\n### Instruction:", "\n### Input:", "\n### Response:"]

def generate_answer(question, max_new_tokens=200):
    """Génère une réponse avec le modèle fine-tuné et mesure la latence."""
    prompt = (
        "### Instruction: Réponds à cette question en français.\n"
        f"### Input: {question}\n"
        "### Response:"
    )
    try:
        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
        start  = time.time()
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,                       # Greedy pour la reproductibilité
                temperature=1.0,                       # Sans effet avec do_sample=False
                eos_token_id=tokenizer.eos_token_id,   # Arrêt à l'EOS token
                pad_token_id=tokenizer.eos_token_id,
            )
        latency_ms = round((time.time() - start) * 1000)

        # Extraction uniquement de la partie générée (après le prompt)
        prompt_len = inputs["input_ids"].shape[1]
        new_tokens = outputs[0][prompt_len:]
        answer     = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

        # Couper dès qu'un marqueur Alpaca apparaît (le modèle recommence un exemple)
        for marker in _ALPACA_STOP_MARKERS:
            if marker in answer:
                answer = answer.split(marker)[0].strip()

        return answer, latency_ms
    except Exception as e:
        print(f"  [ERROR] Génération : {e}")
        return "", 0

print("Modèle en mode inférence. Lancement sur le test set...")

In [ ]:
# Inférence sur tout le test set avec barre de progression
finetuned_predictions = []

for item in tqdm(test_data, desc="Inférence fine-tuné"):
    question    = item.get('question', '')
    true_answer = item.get('answer', '')

    predicted, latency = generate_answer(question)

    finetuned_predictions.append({
        "pair_id":          item.get('pair_id', ''),
        "question":         question,
        "predicted_answer": predicted,
        "true_answer":      true_answer,
        "latency_ms":       latency,
        "method":           "finetuned"
    })

latencies = [p['latency_ms'] for p in finetuned_predictions if p['latency_ms'] > 0]
print(f"\nInférence terminée : {len(finetuned_predictions)} prédictions")
print(f"Latence moyenne    : {np.mean(latencies):.0f} ms" if latencies else "Latence : N/A")

In [ ]:
# Sauvegarde des prédictions du modèle fine-tuné sur Drive
finetuned_path = os.path.join(RESULTS_PATH, 'finetuned_predictions.json')
try:
    with open(finetuned_path, 'w', encoding='utf-8') as f:
        json.dump(finetuned_predictions, f, ensure_ascii=False, indent=2)
    print(f"Prédictions sauvegardées : {finetuned_path}")
    print(f"  → {len(finetuned_predictions)} entrées, {os.path.getsize(finetuned_path)/1024:.1f} Ko")
except Exception as e:
    print(f"[ERROR] Sauvegarde prédictions : {e}")

## 10. Résumé final

In [ ]:
# Affichage complet du résumé de ce qui a été produit
latencies = [p['latency_ms'] for p in finetuned_predictions if p['latency_ms'] > 0]

print("=" * 65)
print("RÉSUMÉ — Notebook 04 : Fine-tuning")
print("=" * 65)
print(f"\nModèle de base      : {BASE_MODEL}")
print(f"LoRA rank / alpha   : {LORA_RANK} / {LORA_ALPHA}")
print(f"Modules cibles      : {TARGET_MODS}")
print(f"Époques             : {NUM_EPOCHS}")
print(f"Exemples entraînem. : {len(hf_dataset)}")
try:
    print(f"Loss finale         : {trainer_stats.training_loss:.4f}")
    print(f"Durée entraînem.    : {train_duration/60:.1f} min")
except Exception:
    pass
print(f"Prédictions test    : {len(finetuned_predictions)}")
print(f"Latence moy. infér. : {np.mean(latencies):.0f} ms" if latencies else "Latence : N/A")

print(f"\nFichiers produits :")
for fpath in [finetuned_path]:
    try:
        print(f"  {fpath}  ({os.path.getsize(fpath)/1024:.1f} Ko)")
    except Exception:
        print(f"  {fpath}")
print(f"  {MODELS_PATH}/  (adaptateur LoRA)")

print("\nAperçu d'une prédiction :")
if finetuned_predictions:
    s = finetuned_predictions[0]
    print(f"  Q      : {s['question'][:80]}")
    print(f"  Prédit : {s['predicted_answer'][:80]}")
    print(f"  Vrai   : {s['true_answer'][:80]}")

print("\n✔ Notebook 04 terminé. Lancez 05_hybrid_eval.ipynb (nécessite aussi un GPU).")
print("=" * 65)